# US Traffic Fatality Prediction — Data Visualization
**DATA 495: Data Science Capstone**  
**Carl Stolpe | UMGC | May 2026**

This notebook produces two key visualizations from the FARS 2015–2016 combined driver-level dataset:

1. **Visualization 1** — Ranked horizontal bar chart: total traffic fatalities by state
2. **Visualization 2** — Temporal bar chart: fatal driver injuries by hour of day

Both visualizations directly informed the feature selection and policy recommendations in the final project.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## 2. Load Prepared Data

In [ ]:
train_df = pd.read_csv('../data/fars_2015_prepared.csv')
val_df   = pd.read_csv('../data/fars_2016_prepared.csv')
df_all   = pd.concat([train_df, val_df], ignore_index=True)

print(f'Combined dataset: {len(df_all):,} driver records')
print(f'Fatal records:    {df_all["INJ_SEV_BINARY"].sum():,} ({df_all["INJ_SEV_BINARY"].mean():.1%})')

## 3. Descriptive Statistics

In [ ]:
cont_vars = ['AGE', 'FATALS', 'HOUR', 'MONTH', 'DRUNK_DR']
desc = df_all[cont_vars].describe().T
desc.index = [
    'Driver Age (years)',
    'Fatalities per Crash',
    'Hour of Crash (0-23)',
    'Month of Crash (1-12)',
    'Drunk Drivers in Crash'
]
desc.index.name = 'Variable'

print('Table 1: Descriptive Statistics (2015-2016 FARS Driver Dataset, N=101,562)')
print(desc[['count','mean','std','min','25%','50%','75%','max']].round(2).to_string())

## 4. Visualization 1 — Total Traffic Fatalities by State

A ranked horizontal bar chart displaying total fatalities for each of the 50 states and DC, with quartile-based color encoding.

In [ ]:
# FIPS state code to abbreviation mapping
state_map = {
    1:'AL', 2:'AK', 4:'AZ', 5:'AR', 6:'CA', 8:'CO', 9:'CT', 10:'DE',
    11:'DC', 12:'FL', 13:'GA', 15:'HI', 16:'ID', 17:'IL', 18:'IN',
    19:'IA', 20:'KS', 21:'KY', 22:'LA', 23:'ME', 24:'MD', 25:'MA',
    26:'MI', 27:'MN', 28:'MS', 29:'MO', 30:'MT', 31:'NE', 32:'NV',
    33:'NH', 34:'NJ', 35:'NM', 36:'NY', 37:'NC', 38:'ND', 39:'OH',
    40:'OK', 41:'OR', 42:'PA', 44:'RI', 45:'SC', 46:'SD', 47:'TN',
    48:'TX', 49:'UT', 50:'VT', 51:'VA', 53:'WA', 54:'WV', 55:'WI', 56:'WY'
}

# Aggregate total fatalities per state across 2015-2016
state_fatals = (
    df_all[df_all['INJ_SEV_BINARY'] == 1]
    .groupby('STATE')
    .size()
    .reset_index(name='Total_Fatalities')
)
state_fatals['State'] = state_fatals['STATE'].map(state_map)
state_fatals = state_fatals.dropna(subset=['State']).sort_values('Total_Fatalities')

# Quartile color encoding
q1, q2, q3 = state_fatals['Total_Fatalities'].quantile([0.25, 0.50, 0.75])
def get_color(val):
    if val >= q3:   return '#8B0000'
    elif val >= q2: return '#CD5C5C'
    elif val >= q1: return '#F08080'
    else:           return '#FFB6C1'

colors = [get_color(v) for v in state_fatals['Total_Fatalities']]

fig, ax = plt.subplots(figsize=(12, 14))
bars = ax.barh(state_fatals['State'], state_fatals['Total_Fatalities'],
               color=colors, edgecolor='white', height=0.8)

# Label top quartile bars
for bar, val in zip(bars, state_fatals['Total_Fatalities']):
    if val >= q3:
        ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height() / 2,
                f'{int(val):,}', va='center', fontsize=8)

legend_patches = [
    mpatches.Patch(color='#8B0000', label='Top 25%'),
    mpatches.Patch(color='#CD5C5C', label='50th-75th pct'),
    mpatches.Patch(color='#F08080', label='25th-50th pct'),
    mpatches.Patch(color='#FFB6C1', label='Bottom 25%')
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
ax.set_xlabel('Total Traffic Fatalities (2015-2016)', fontsize=11)
ax.set_title('Total Traffic Fatalities by State\nFARS Dataset 2015-2016',
             fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/fatalities_by_state.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

### Visualization 1 Findings

- Texas and California alone account for approximately 20% of all fatalities across 2015–2016
- The top 5 states (TX, CA, FL, GA, NC) account for over 36% of all fatal crash deaths
- The bottom quartile states are all small, densely urbanized jurisdictions (DC, RI, VT)
- The 147-fold difference between Texas and DC reflects population, road network size, and structural policy differences
- State was subsequently confirmed as the 10th most important predictor by the SHAP analysis (mean |SHAP| = 0.0106)

## 5. Visualization 2 — Fatal Driver Injuries by Hour of Day

A temporal bar chart showing the distribution of fatal driver injuries across the 24-hour day, with color encoding for four time blocks.

In [ ]:
# Filter to fatal records with known hour
fatal_df = df_all[
    (df_all['INJ_SEV_BINARY'] == 1) &
    (df_all['HOUR'].notna()) &
    (df_all['HOUR'] < 24)
].copy()

fatal_df['HOUR'] = fatal_df['HOUR'].astype(int)
hourly_counts = fatal_df.groupby('HOUR').size().reindex(range(24), fill_value=0)

# Time block color mapping
def hour_color(h):
    if   0  <= h < 6:  return '#1a237e'  # Overnight — dark blue
    elif 6  <= h < 12: return '#e65100'  # Morning   — orange
    elif 12 <= h < 18: return '#2e7d32'  # Afternoon — green
    else:              return '#6a1b9a'  # Evening   — purple

colors = [hour_color(h) for h in range(24)]
labels = [f'{h:02d}:00' for h in range(24)]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(24), hourly_counts.values, color=colors, edgecolor='white', width=0.8)

ax.set_xticks(range(24))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Hour of Day (24-Hour Clock)', fontsize=11)
ax.set_ylabel('Number of Fatal Driver Injuries', fontsize=11)
ax.set_title('Fatal Driver Injuries by Hour of Day\nFARS Dataset 2015-2016 Combined',
             fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', alpha=0.3)

legend_patches = [
    mpatches.Patch(color='#1a237e', label='Overnight (12am-6am)'),
    mpatches.Patch(color='#e65100', label='Morning (6am-12pm)'),
    mpatches.Patch(color='#2e7d32', label='Afternoon (12pm-6pm)'),
    mpatches.Patch(color='#6a1b9a', label='Evening (6pm-12am)')
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/fatalities_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

### Visualization 2 Findings

- The three highest-fatality hours are 4 PM (2,659), 3 PM (2,593), and 5 PM (2,581)
- The afternoon block (12pm–6pm) accounts for ~31% of all hourly fatal injuries despite covering only one-sixth of the 24-hour cycle
- A secondary overnight peak (midnight–3 AM) reflects bar-closing hours and impaired driving
- The peak-to-trough ratio is approximately 3.5:1
- Hour of crash was subsequently confirmed as the 7th most important predictor by the SHAP analysis (mean |SHAP| = 0.0174)

## 6. Cross-Validation with Model Results

Both visualization findings were validated by the SHAP analysis:

| Visual Finding | SHAP Rank | Mean |SHAP Value| |
|---|---|---|
| State geographic concentration | 10 | 0.0106 |
| Hour of crash temporal pattern | 7 | 0.0174 |

The alignment between visualization findings and SHAP feature importance results confirms that the geographic and temporal risk structures are genuine features of the underlying crash data rather than visualization artifacts.